In [ ]:
import xarray as xr
import numpy as np

begin_year = 2005
fn = (
    "/rd/cenfic3/cenmod/era40/vortex/s2m/alp_allslopes/reanalysis_2020.2/meteo/FORCING_"
    + str(begin_year)
    + "080106_"
    + str(begin_year + 1)
    + "080106.nc"
)
local_data = xr.open_dataset(fn)

In [2]:
fn = "FORCING_reanalysis2020.2@lafaysse_alp_allslopes_2005080106_2006080106.nc"
hendrix_data = xr.open_dataset(fn)

In [ ]:
hendrix_data.to_netcdf

In [62]:
local_data

<xarray.Dataset> Size: 2GB
Dimensions:          (Number_of_points: 4471, time: 8761, massif: 24)
Coordinates:
  * time             (time) datetime64[ns] 70kB 2005-08-01T06:00:00 ... 2006-...
  * massif           (massif) float64 192B 1.0 2.0 3.0 4.0 ... 22.0 23.0 30.0
Dimensions without coordinates: Number_of_points
Data variables: (12/24)
    ZS               (Number_of_points) float32 18kB 300.0 300.0 ... 3e+03 3e+03
    aspect           (Number_of_points) float32 18kB ...
    slope            (Number_of_points) float32 18kB ...
    massif_number    (Number_of_points) float64 36kB ...
    PSurf            (time, Number_of_points) float32 157MB ...
    Tair             (time, Number_of_points) float32 157MB ...
    ...               ...
    CO2air           (time, Number_of_points) float32 157MB ...
    FRC_TIME_STP     float32 4B ...
    LAT              (Number_of_points) float64 36kB 46.18 46.18 ... 46.39 46.39
    LON              (Number_of_points) float64 36kB 6.645 6.645 ... 7.303 7.303
    isoZeroAltitude  (time, massif) float64 2MB ...
    rainSnowLimit    (time, massif) float64 2MB ...
Attributes: (12/45)
    id:                            s2m_reanalysis_2020.2
    references:                    https://doi.org/10.1175/2008JAMC1808.1;htt...
    date_created:                  2021-04-30T10:01:00
    platform:                      MODELS_ANALYSES.REANALYSIS_MODELS
    platform_vocabulary:           GCMD,AERIS
    contributor_name:              Matthieu Vernay ; Matthieu Lafaysse
    ...                            ...
    publisher_url:                 https://www.umr-cnrm.fr/spip.php?rubrique85
    creator_url:                   https://www.umr-cnrm.fr/spip.php?rubrique85
    safran_tag:                    s2m_reanalysis_2020.2
    title:                         S2M reanalysis: meteorological variables
    summary:                       This file takes part from a 61-years reana...
    keywords:                      MOUNTAINS,INCOMING SOLAR RADIATION,LONGWAV...

In [ ]:
# Fit Dickinson to data
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

n_tot = len(x_poste)
fig, axs = plt.subplots(n_tot, 1, figsize=(15, 3 * n_tot))


def preprocess_virrs(image_sat: xr.DataArray) -> xr.DataArray:
    return image_sat.where(image_sat <= 200) / 200


for pp in range(n_tot):
    ax = axs[pp]
    xi, yi = x_poste[pp], y_poste[pp]

    fsc = viirs.sel(x=xi, y=yi, method="nearest").dropna(dim="time")
    if len(fsc) == 0:
        continue
    sd = edel.sel(x=xi, y=yi, method="nearest", time=fsc.coords["time"]).median(dim="member")
    coeff, cov = curve_fit(dickinson, xdata=sd.values, ydata=fsc.values, sigma=0.25, bounds=([1.1, 0.09], [1.7, 0.13]))
    ai, bi = coeff

    # Members
    dickinson(sd=edel.sel(x=xi, y=yi, method="nearest"), a=ai, b=bi).plot.line(x="time", ax=ax, linewidth=0.5)
    # Median member
    dickinson(sd=edel.sel(x=xi, y=yi, method="nearest"), a=ai, b=bi).median(dim="member").plot.line(
        x="time", ax=ax, color="indigo", linestyle="dashed", linewidth=3
    )
    # VIIRS
    viirs.sel(x=xi, y=yi, method="nearest").plot.line(x="time", ax=ax, color="black", linestyle="dashed", linewidth=3)
    # Pléiades
    dickinson(sd=pleiades.sel(x=xi, y=yi, method="nearest"), a=ai, b=bi).plot.line(
        x="time", ax=ax, marker="*", linewidth=0, markersize=15, color="gold"
    )
    # BD Clim
    dickinson(sd=bdclim.where(bdclim["x"] == xi, drop=True).where(bdclim["y"] == yi, drop=True), a=ai, b=bi).plot.line(
        x="time", ax=ax, linestyle="dashed", linewidth=3, color="chocolate"
    )
    # pleiades.sel(x=xi, y=yi, method="nearest").sel(time=slice("2022-01-01", "2022-06-01")).data_vars[
    #     "snow_cover_fraction"
    # ].plot.line(x="time", ax=ax, marker="*", linewidth=0, markersize=15, color="gold")
    # # BD Clim
    # bdclim.where(bdclim["x"] == xi, drop=True).where(bdclim["y"] == yi, drop=True).sel(
    #     time=slice("2022-01-01", "2022-06-01")
    # ).data_vars["snow_cover_fraction"].plot.line(x="time", ax=ax, linestyle="dashed", linewidth=3, color="chocolate")
    ax.grid(True)
    ax.set_title(f"a={ai:.2f}, b={bi:.2f}")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(20, 10))


def dickinson_arr(sd: np.ndarray, a: np.ndarray, b: np.ndarray):
    return np.minimum(1, (a * sd) / (sd + b))


for i, acq in enumerate(pleiades.time):
    err_viirs = viirs.sel(time=acq, method="nearest") - dickinson_arr(sd=pleiades.sel(time=acq), a=res.a, b=res.b)
    im = axs[i][0].imshow(err_viirs, vmin=-1, vmax=1, cmap="coolwarm")
    axs[i][0].set_title(f"time = {acq.values}, VIIRS FSC - PLEIADES FSC")
    err_edel = dickinson_arr(edel.sel(time=acq, method="nearest").median(dim="member"), a=res.a, b=res.b) - dickinson_arr(
        sd=pleiades.sel(time=acq), a=res.a, b=res.b
    )
    axs[i][1].imshow(err_edel, vmin=-1, vmax=1, cmap="coolwarm")
    axs[i][1].set_title(f"time = {acq.values}, EDELWEISS FSC - PLEIADES FSC")
# fig.colorbar(im,ax=axs[0])